In [1]:


import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://www.kanker.nl"
START_URL = "https://www.kanker.nl/kankersoorten"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_cancer_links():
    """Extracts all specific unique cancer links using the main AZ index container."""
    print(f"Fetching overview page: {START_URL}")
    response = requests.get(START_URL, headers=HEADERS)
    if response.status_code != 200:
        print(f"Failed to load overview page. Status code: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    cancer_links = {}

    # Target the primary layout grid list where alphabetical diseases are housed
    grid_container = soup.find('div', class_='dynamic-grid') or soup.find('main')
    
    if not grid_container:
        grid_container = soup # fallback if structure alters

    for link in grid_container.find_all('a', href=True):
        href = link['href']
        # Extract links formatted like /kankersoorten/alvleesklierkanker
        if href.startswith('/kankersoorten/') and len(href.split('/')) == 3:
            name = link.get_text(strip=True)
            full_url = BASE_URL + href
            
            # Avoid picking up generic tracking shortcuts or duplicate labels
            if name and "bekijk alle" not in name.lower() and "meer kankersoorten" not in name.lower():
                if name not in cancer_links:
                    cancer_links[name] = full_url

    print(f"Found {len(cancer_links)} unique cancer variants to process.")
    return cancer_links

def scrape_cancer_details(cancer_name, hub_url):
    """Navigates to subpages and extracts structural parameters securely."""
    slug = hub_url.split('/')[-1]
    target_url = f"{hub_url}/algemeen/wat-is-{slug}"
    
    print(f"Scraping detailed info for [{cancer_name}] via {target_url}...")
    
    data_row = {
        "Cancer Type": cancer_name,
        "Source URL": target_url,
        "General Description": "N/A",
        "Prognosis (Vooruitzichten)": "N/A",
        "Symptoms (Symptomen)": "N/A",
        "Causes (Oorzaken)": "N/A",
        "Treatments (Behandeling)": "N/A"
    }
    
    response = requests.get(target_url, headers=HEADERS)
    if response.status_code != 200:
        target_url = hub_url
        response = requests.get(target_url, headers=HEADERS)
        if response.status_code != 200:
            return data_row

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # --- FIXED GENERAL DESCRIPTION EXTRACTION WITH SYSTEM NOISE FILTER ---
    main_article = soup.find('article') or soup.find('main') or soup
    
    intro_paragraphs = []
    skip_phrases = [
        "deze informatie is gecontroleerd", 
        "naar colofon", 
        "printen", 
        "opslaan", 
        "delen",
        "luister"
    ]
    
    for p in main_article.find_all('p'):
        p_text = p.get_text(strip=True)
        if not p_text:
            continue
        if "lees op deze pagina" in p_text.lower():
            break
        if any(phrase in p_text.lower() for phrase in skip_phrases):
            continue
        intro_paragraphs.append(p_text)
        
    if intro_paragraphs:
        data_row["General Description"] = intro_paragraphs[0]

    # --- SUBSECTIONS HEADER MAPPING ---
    headings = soup.find_all(['h2', 'h3'])
    for heading in headings:
        heading_text = heading.get_text(strip=True).lower()
        
        content_paragraphs = []
        next_node = heading.next_sibling
        while next_node:
            if next_node.name in ['h2', 'h3']:
                break
            if next_node.name in ['p', 'ul', 'ol']:
                content_paragraphs.append(next_node.get_text(" ", strip=True))
            next_node = next_node.next_sibling
            
        section_content = " ".join(content_paragraphs)
        if not section_content:
            continue
            
        if 'vooruitzichten' in heading_text:
            data_row["Prognosis (Vooruitzichten)"] = section_content
        elif 'symptomen' in heading_text or 'klachten' in heading_text:
            data_row["Symptoms (Symptomen)"] = section_content
        elif 'oorzaken' in heading_text or 'risicofactoren' in heading_text:
            data_row["Causes (Oorzaken)"] = section_content
        elif 'behandeling' in heading_text:
            data_row["Treatments (Behandeling)"] = section_content

    return data_row

def main():
    cancer_directory = get_cancer_links()
    if not cancer_directory:
        print("No paths discovered. Check structural layout rules.")
        return

    all_cancer_data = []
    
    # Process items inside the clean directory mapping
    for name, url in cancer_directory.items():
        try:
            row_data = scrape_cancer_details(name, url)
            all_cancer_data.append(row_data)
        except Exception as e:
            print(f"Error scraping {name}: {e}")
            
        time.sleep(1.5)  # Throttling protection

    # Save to table files
    df = pd.DataFrame(all_cancer_data)
    df.to_csv("kanker_nl_master_table.csv", index=False, encoding='utf-8-sig')
    df.to_excel("kanker_nl_master_table.xlsx", index=False)
    print("\nMaster table files generated successfully with correct isolated links!")

if __name__ == "__main__":
    main()

Fetching overview page: https://www.kanker.nl/kankersoorten
Found 73 unique cancer variants to process.
Scraping detailed info for [Alvleesklierkanker] via https://www.kanker.nl/kankersoorten/alvleesklierkanker/algemeen/wat-is-alvleesklierkanker...
Scraping detailed info for [Anuskanker] via https://www.kanker.nl/kankersoorten/anuskanker/algemeen/wat-is-anuskanker...
Scraping detailed info for [Atypisch fibroxanthoom] via https://www.kanker.nl/kankersoorten/atypisch-fibroxanthoom/algemeen/wat-is-atypisch-fibroxanthoom...
Scraping detailed info for [Baarmoederhalskanker] via https://www.kanker.nl/kankersoorten/baarmoederhalskanker/algemeen/wat-is-baarmoederhalskanker...
Scraping detailed info for [Baarmoederkanker] via https://www.kanker.nl/kankersoorten/baarmoederkanker/algemeen/wat-is-baarmoederkanker...
Scraping detailed info for [Baarmoedersarcoom] via https://www.kanker.nl/kankersoorten/baarmoedersarcoom/algemeen/wat-is-baarmoedersarcoom...
Scraping detailed info for [Basaalcelcarc